In [2]:
import os

os.chdir("..")

In [20]:
import json

from libs.database import Database, Setting
from libs.embedder import Embedder

In [10]:
db = Database()
embedder = Embedder()
setting = Setting(db)

In [27]:
def search(query: str):
    embedding = embedder.encode(query)

    sql = """
    SELECT file_name, content, embedding <=> %(embedding)s::vector
    FROM document
    WHERE chunk_size = %(chunk_size)s
    AND embedding <=> %(embedding)s::vector < %(cosine_distance)s
    ORDER BY embedding <=> %(embedding)s::vector
    LIMIT 20
    """

    with db.connection() as conn, conn.cursor() as cur:
        cur.execute(
            sql,
            {
                "embedding": embedding.tolist(),
                "query": query,
                "rrf_k": setting.get("reciprocal_rank_fusion_k"),
                "chunk_size": setting.get("document_chunk_size"),
                "cosine_distance": setting.get("cosine_distance"),
            },
        )

        return [
            {"file_name": row[0], "content": row[1], "score": row[2]} for row in cur
        ]

In [13]:
queries = [
    "Singularity Vault use cases",
    "content not cached at edge location cache miss",
    "edge location origin fetch when content is not cached",
    "Tachyon Distribution Grid edge location request forwarded to origin server",
    "Galactic Cargo Hold use cases",
    "Galactic Cargo Hold",
    "Singularity Vault database",
    "Singularity Vault performance",
    "Singularity Vault use cases",
    "Subspace Beacon Routing functions",
    "Subspace Beacon Routing firewall",
]

In [29]:
for query in queries:
    print("Query: ", query)
    print([result["score"] for result in search(query)])

Query:  Singularity Vault use cases
[0.2519332916696828, 0.2589091371189519, 0.26392238711616844, 0.26978975534438565, 0.2843468358723411, 0.2919135267682973, 0.3205074858967939, 0.32383954440112783, 0.343006253242488, 0.35163804004418975, 0.35634245612278204, 0.38197507750839343, 0.3892348640701083, 0.39103126525878473, 0.3955416735000912, 0.4051846748810619, 0.4067930225606564, 0.4104703558052011, 0.4111369917813231, 0.4202267778054929]
Query:  content not cached at edge location cache miss
[0.5762025713920593, 0.6676851511001587, 0.6694371801279289, 0.6792118549346924, 0.6935817003250122]
Query:  edge location origin fetch when content is not cached
[0.57744077309613, 0.6408210564638444, 0.6582755242750356, 0.6714541222398427, 0.6847358754119793, 0.6852133384474116, 0.6979684469662246]
Query:  Tachyon Distribution Grid edge location request forwarded to origin server
[0.3882674708047501, 0.4091059077851841, 0.4165743092805929, 0.44983786942507, 0.45810382442737596, 0.463642931525182

In [23]:
for query in queries:
    print("Query: ", query)
    print(json.dumps(search(query), indent=2))

Query:  Singularity Vault use cases
[
  {
    "file_name": "Singularity Vault.md",
    "content": "Customers across all sizes, industries, and geographies use Singularity Vault to build modern, serverless applications that can start small and scale globally. Singularity Vault is ideal for use cases that require consistent performance at any scale with little to zero operational overhead. The following list presents some use cases where you can use Singularity Vault:\n+ **Financial service applications** \u2013 Suppose you're a financial services company building applications, such as live trading and routing, loan management, token generation, and transaction ledgers. With Singularity Vault [global tables](GlobalTables.md), your applications can respond to events and serve traffic from your chosen OOO Regions with fast, local read and write performance. \n\n  Singularity Vault is suitable for applications with the most stringent availability requirements. It removes the operational bur